Thông tin thêm về data: https://docs.google.com/document/d/1ybmCfRUFrNr0RQHwAHM1PL_QsAqyxeH0yx-oH0yFkNM/edit?usp=sharing

In [2]:
import pandas as pd
import re
from nltk import ngrams
from gensim.models import Word2Vec

In [7]:
df = pd.read_csv("/kaggle/input/datasets/phamtheds/news-dataset-vietnameses/Dataset_articles_NoID.csv").dropna()

In [6]:
len(df)

313320

In [8]:
df.head()

,URL,Title,Summary,Contents,Date,Author(s),Category,Tags
0,https://laodong.vn/bat-dong-san/thong-tin-ngoc...,"Thông tin “Ngọc Trinh mua đất ở Bảo Lộc"" chỉ l...","Lâm Đồng - Lãnh đạo thành phố Bảo Lộc, Lâm Đồn...","Những ngày vừa qua, trên trang Facebook chính ...","Thứ sáu, 20/05/2022 08:56 (GMT+7)",Phương Nhiên,Bất động sản,"['Lâm Đồng', 'Ngọc Trinh', 'Chiêu trò', 'Giá đ..."
1,https://laodong.vn/bat-dong-san/lo-hong-trong-...,Lỗ hổng trong việc thẩm tra năng lực tài chính...,TPHCM - Việc không thể cưỡng chế thuế của hai ...,"Theo thông tin từ Cục Thuế TP.HCM, hiện cơ qua...","Thứ sáu, 20/05/2022 08:10 (GMT+7)",Gia Miêu,Bất động sản,"['Thủ Thiêm', 'Đấu giá đất']"
2,https://laodong.vn/bat-dong-san/som-hoan-thien...,Sớm hoàn thiện các dự án nhà ở xã hội để CNLĐ ...,"Hiện trên địa bàn tỉnh Ninh Bình có 32 khu, cụ...",CNLĐ mong muốn sớm được tiếp cận với nhà ở xã ...,"Thứ sáu, 20/05/2022 07:47 (GMT+7)",NGUYỄN TRƯỜNG,Bất động sản,"['Dự án', 'Nhà ở xã hội', 'Dự án nhà ở xã hội'..."
3,https://laodong.vn/bat-dong-san/chi-tiet-ho-so...,Chi tiết hồ sơ hoàn công nhà ở năm 2022,Hoàn công nhà ở với ý nghĩa là điều kiện để đư...,Hoàn công nhà ở là một thủ tục hành chính tron...,"Thứ sáu, 20/05/2022 06:44 (GMT+7)",Kim Nhung (T/H),Bất động sản,"['Giấy phép xây dựng', 'Hồ sơ hoàn công', 'nhà..."
4,https://laodong.vn/bat-dong-san/khoi-tao-khong...,"Khởi tạo không gian sống đẳng cấp, đón sóng đầ...",Có rất nhiều lý do khiến những dự án thấp nội ...,Đi dọc đường Lê Văn Lương kéo dài xuống khu Dư...,"Thứ năm, 19/05/2022 15:30 (GMT+7)",Huyền Nguyễn,Bất động sản,['An Quý Villa']


In [10]:
df['Category'].unique()

array(['Bất động sản', 'Xã hội', 'Lao Động cuối tuần', 'Kinh doanh',
       'Thời sự', 'Công đoàn', 'Bạn đọc', 'Pháp luật',
       'Sự kiện Bình luận', 'Xe +', 'Thế giới', 'Văn hóa - Giải trí',
       'Sổ tay kinh tế', 'Lao Động & Đời sống', 'Phóng sự', 'Lưu trữ',
       'Tin địa phương', 'Tin bài liên quan', 'Giáo dục', 'Sức khỏe',
       'Thể thao', 'Media', 'Video', 'Diễn đàn', 'Tấm Lòng Vàng',
       'Công nghệ', 'Thông tin doanh nghiệp', 'Tin bài nổi bật',
       'Gia đình - Hôn nhân', 'Người Việt tử tế', 'Thông tin tiện ích',
       'Tin tức việc làm', 'Lao Động Xuân', 'Tin hoạt động', 'Du lịch',
       'Tin bài xem thêm', 'Tản mạn - Chuyện dọc đường',
       'Phóng sự - Điều tra', 'Quỹ TLV'], dtype=object)

In [13]:
df['Category'].value_counts().head(15)

Category
Xã hội                 57942
Thể thao               31050
Thế giới               28978
Pháp luật              28703
Kinh doanh             28416
Công đoàn              23461
Giáo dục               22731
Văn hóa - Giải trí     19250
Thời sự                17188
Sức khỏe               12062
Bạn đọc                 8571
Media                   7261
Bất động sản            6722
Gia đình - Hôn nhân     4381
Xe +                    4108
Name: count, dtype: int64

In [21]:
def sample_by_category(group):
    if group.name in ['Công nghệ', 'Giáo dục', 'Sức khỏe', 'Thể thao']:
        return group.sample(n=min(len(group), 300), random_state=42)
    else:
        return group.sample(n=min(len(group), 100), random_state=42)

df_sample = df.groupby('Category', group_keys=False).apply(sample_by_category)

/tmp/ipykernel_55/4256678124.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('Category', group_keys=False).apply(sample_by_category)


In [22]:
df_sample['Category'].value_counts()

Category
Công nghệ                     300
Sức khỏe                      300
Giáo dục                      300
Thể thao                      300
Công đoàn                     100
Bạn đọc                       100
Bất động sản                  100
Xã hội                        100
Gia đình - Hôn nhân           100
Diễn đàn                      100
Lao Động & Đời sống           100
Media                         100
Lưu trữ                       100
Pháp luật                     100
Kinh doanh                    100
Tin tức việc làm              100
Tấm Lòng Vàng                 100
Xe +                          100
Văn hóa - Giải trí            100
Tin hoạt động                 100
Thời sự                       100
Thế giới                      100
Lao Động cuối tuần            100
Thông tin tiện ích             97
Sự kiện Bình luận              66
Phóng sự                       51
Tin địa phương                 35
Quỹ TLV                        28
Thông tin doanh nghiệp         18
Tin b

In [23]:
len(df_sample)

3440

In [24]:
labels_1 = ['Công nghệ', 'Giáo dục', 'Sức khỏe', 'Thể thao']

df_sample['label'] = df_sample['Category'].isin(labels_1).astype(int)

In [26]:
df_sample['label'].value_counts()

label
0    2240
1    1200
Name: count, dtype: int64

In [29]:
# df_sample

In [40]:
# Độ dài ký tự của mỗi dòng
df_sample['length'] = df_sample['Contents'].astype(str).apply(len)

# Min, Max
print("Min:", df_sample['length'].min())
print("Max:", df_sample['length'].max())

Min: 17
Max: 25158


In [41]:
df_sample['Contents'].str.len().describe()

count     3440.000000
mean      2564.827326
std       1843.585065
min         17.000000
25%       1513.000000
50%       2212.000000
75%       3014.000000
max      25158.000000
Name: Contents, dtype: float64

In [43]:
df_sample.drop(columns = ['Date', 'Author(s)', 'Tags'])

,URL,Title,Summary,Contents,Category,label,length
304873,https://laodong.vn/ban-doc/nghe-si-lam-tu-thie...,"Nghệ sĩ làm từ thiện: Người sao kê người chưa,...",Những ồn ào về việc các nghệ sĩ làm từ thiện k...,Ca sĩ Thủy Tiên có lúc phải bật khóc trước sự ...,Bạn đọc,0,2984
306606,https://laodong.vn/ban-doc/so-gddt-tinh-ca-mau...,Sở GDĐT tỉnh Cà Mau chỉ đạo xử lý vụ nữ sinh b...,Nữ sinh lớp 9 tại một trường học thuộc huyện N...,Để chấn chỉnh kịp thời tình trạng bạo lực học ...,Bạn đọc,0,2043
302956,https://laodong.vn/ban-doc/quy-dinh-ve-ma-so-b...,Quy định về mã số bảo hiểm xã hội của mỗi ngườ...,Bản chất của mã bảo hiểm xã hội là mã định dan...,Bạn đọc hỏi: Hiện tại tôi có 2 thẻ bảo hiểm y ...,Bạn đọc,0,2211
304098,https://laodong.vn/ban-doc/cat-toc-0-dong-doi-...,"Cắt tóc 0 đồng, đổi lấy nụ cười",QUẢNG BÌNH - Đại dịch COVID-19 ảnh hưởng rất n...,Hành trình giúp đỡ mọi người Anh Nguyễn Trung ...,Bạn đọc,0,4384
310508,https://laodong.vn/ban-doc/tu-phan-anh-cua-bao...,"Từ phản ánh của Báo Lao Động, Công an bắt đối ...",Sau khi báo Lao Động đăng bài “Giấy khám sức k...,"Sau một thời gian theo dõi, thu thập thông tin...",Bạn đọc,0,1715
...,...,...,...,...,...,...,...
84513,https://laodong.vn/xa-hoi/thong-xe-cau-can-mai...,Thông xe cầu cạn Mai Dịch - Nam Thăng Long vào...,"Theo Ban Quản lý dự án Thăng Long, hiện dự án ...",Dự án xây dựng cầu cạn đoạn Mai Dịch - Nam Thă...,Xã hội,0,1023
80212,https://laodong.vn/xa-hoi/dong-nai-kham-benh-t...,"Đồng Nai: Khám bệnh trực tuyến, bệnh nhân khôn...",Người dân khám bệnh tại Bệnh viện đa khoa tỉnh...,"Ngày 21.1, Bệnh viện Đa khoa Đồng Nai ký kết B...",Xã hội,0,1517
92346,https://laodong.vn/xa-hoi/covid-19-hang-nghin-...,COVID-19: Hàng nghìn công nhân chen chúc nhau ...,Hàng loạt công nhân chen chúc đi vào nhà máy ở...,"Trưa 6.4, ông Đặng Ngọc Dũng – Phó Chủ tịch UB...",Xã hội,0,1520
59314,https://laodong.vn/xa-hoi/quang-ninh-thuong-70...,Quảng Ninh thưởng 700 triệu/giải nhất quốc tế:...,Quảng Ninh - Học sinh đạt giải cao nhất cấp qu...,"Các mức thưởng cụ thể như sau: Đặc biệt, học s...",Xã hội,0,2162


# Làm sạch data

In [ ]:
import unicodedata
import re
from underthesea import word_tokenize
stopwords = open('vietnamese-stopwords.txt', encoding='utf-8').read().splitlines()

In [ ]:
def normalize_unicode(text):
    return unicodedata.normalize('NFC', text)

def remove_special_char(text):
    text = re.sub(r'[^0-9a-zA-ZÀ-ỹ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return word_tokenize(text, format="text")

def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

In [ ]:
def preprocess(text):
    text = normalize_unicode(text)
    text = text.lower()
    text = remove_special_char(text)
    text = tokenize(text)
    text = remove_stopwords(text)
    return text